In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, accuracy_score

# 1. Carregar os dados 
# Usamos errors='replace' para garantir que caracteres zoados virem '?' e possamos consertá-los
df = pd.read_csv('avaliar.csv', sep=';', encoding='utf-8')

# 2. Dicionário de Correção Definitivo
# Ele vai caçar padrões com "?" ou caracteres zoados e substituir pelo acento correto
correcoes = {
    r'N�o Identificado': 'Não Identificado',
    r'N�o Identificado': 'Não Identificado', # Pega qualquer variação (N_o, N~o)
    r'Homic�dio': 'Homicídio',
    r'Tr�fico': 'Tráfico',
    r'Les�o': 'Lesão',
    r'Com�rcio': 'Comércio',
    r'Resid�ncia': 'Residência',
    r'Eletr�nico': 'Eletrônico',
    r'Condu�o': 'Condução',
    r'ap�s': 'após',
    r'Extors�o': 'Extorsão',
    r'Rel�mpago': 'Relâmpago',
    r'Usu�rio': 'Usuário',
    r'Estelionat�rio': 'Estelionatário',
    r'Extorsion�rio': 'Extorsionário',
    r'Pol�ciais': 'Policiais',
    r'Pol�tico': 'Político',
    r'Ve�culos': 'Veículos',
    r'Ve�culo': 'Veículo',
    r'Cad�ver': 'Cadáver',
    r'Mil�cia': 'Milíica',
    r'Amea�a': 'Ameaça',
    r'Latroc�nio': 'Latrocínio',
    r'Tr�fico': 'Tráfico',
    r'Morte por Interven��o Policial': 'Morte Por Intervenção Policial',
    r'Policiais Mortos em Servi�o': 'Policiais Mortos em Serviço'
}

colunas_analise = [
    'nome_autor', 'acertou_autor', 
    'nome_grupo', 'acertou_nome_grupo', 
    'nome_crime', 'acertou_nome_crime'
]

# 3. Limpeza e Aplicação das Correções
for col in colunas_analise:
    if col in df.columns:
        # Preenche vazios
        df[col] = df[col].fillna('Não Identificado').astype(str).str.strip()
        df[col] = df[col].replace({'nan': 'Não Identificado', '': 'Não Identificado', 'None': 'Não Identificado'})
        
        # Aplica as correções de acentuação usando Regex (expressões regulares)
        for erro, acerto in correcoes.items():
            df[col] = df[col].str.replace(erro, acerto, regex=True)

# 4. Função para calcular, plotar e salvar a Matriz de Confusão
def plot_cm(y_true, y_pred, title, filename):
    labels = sorted(list(set(y_true) | set(y_pred)))
    cm = confusion_matrix(y_true, y_pred, labels=labels)
    
    plt.figure(figsize=(14, 12)) 
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels)
    
    plt.title(f'Matriz de Confusão - {title}', fontsize=16, pad=20)
    plt.ylabel('Valor Real', fontsize=14)
    plt.xlabel('Classificação do Sabiá', fontsize=14)
    plt.xticks(rotation=45, ha='right', fontsize=10)
    plt.yticks(rotation=0, fontsize=10)
    plt.tight_layout()
    plt.savefig(filename, dpi=300)
    plt.close()

# 5. Cálculos de Acurácia e Geração dos Gráficos
acc_autor = accuracy_score(df['acertou_autor'], df['nome_autor'])
plot_cm(df['acertou_autor'], df['nome_autor'], 'Autor', 'cm_autor.png')

acc_grupo = accuracy_score(df['acertou_nome_grupo'], df['nome_grupo'])
plot_cm(df['acertou_nome_grupo'], df['nome_grupo'], 'Grupo Criminoso', 'cm_grupo.png')

acc_crime = accuracy_score(df['acertou_nome_crime'], df['nome_crime'])
plot_cm(df['acertou_nome_crime'], df['nome_crime'], 'Tipo de Crime', 'cm_crime.png')

# Bairro
if 'acertou_bairro' in df.columns:
    bairro_matches = df['acertou_bairro'].astype(str).str.lower().str.strip() == 'sim'
    acc_bairro = bairro_matches.mean()
else:
    acc_bairro = 0.0

# 6. Resultados
print("="*30)
print("     RESULTADOS DE ACURÁCIA")
print("="*30)
print(f"Acurácia Autor:   {acc_autor:.2%}")
print(f"Acurácia Grupo:   {acc_grupo:.2%}")
print(f"Acurácia Crime:   {acc_crime:.2%}")
print(f"Acurácia Bairro:  {acc_bairro:.2%}")
print("="*30)